In [ ]:
from pathlib import Path

import pandas as pd

csv_path = "MultiForceProbing.csv"
data = pd.read_csv(csv_path)
data.head()


In [ ]:
import re

eit_cols = [c for c in data.columns if re.fullmatch(r"eit_\d+", c)]
eit_labels = [c.removeprefix("eit_") for c in eit_cols]
eit_col_by_index = {int(label): col for label, col in zip(eit_labels, eit_cols)}

print(f"Loaded {len(data)} samples with {len(eit_cols)} EIT channels")


In [ ]:
import matplotlib.pyplot as plt


reference_row = 1

plt.figure(figsize=(12, 4))
plt.plot(data.loc[reference_row, eit_cols].to_numpy())

plt.xticks(
    ticks=range(len(eit_cols)),
    labels=eit_labels,
    rotation=90
)

plt.xlabel("Channel")
plt.ylabel("Voltage (V)")
plt.title(f"EIT channel vector at row {reference_row}")
plt.tight_layout()

plt.savefig(f"../../Figures/RefSig{Path(csv_path).stem}.svg", format="svg", bbox_inches="tight")
plt.show()


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


raw_data = pd.read_csv("MultiForceProbing.csv)
print(f"Loaded {len(raw_data)} rows from {csv_path}")
raw_data.head()

In [ ]:
import re


def sorted_eit_columns(columns):
    numbered = []
    for col in columns:
        match = re.fullmatch(r"eit_(\d+)", col)
        if match:
            numbered.append((int(match.group(1)), col))
    return [col for _, col in sorted(numbered)]


def prepare_repeated_contact_frames(frame):
    frame = frame.copy()
    eit_cols = sorted_eit_columns(frame.columns)
    if not eit_cols:
        raise ValueError("No eit_<index> columns found in the CSV.")

    time_column = next(
        (col for col in ["sensor_time_s", "eit_time_s", "phase_elapsed_s"] if col in frame.columns),
        None,
    )
    if time_column is None:
        raise KeyError("Expected one of sensor_time_s, eit_time_s, or phase_elapsed_s for plotting.")

    if "row_type" in frame.columns:
        row_type = frame["row_type"].astype(str).str.lower()
        eit_frame = frame[row_type.isin(["baseline_eit", "eit"])].copy()
        force_frame = frame[row_type.eq("force")].copy()
    else:
        has_eit_values = frame[eit_cols].notna().any(axis=1)
        eit_frame = frame[has_eit_values].copy()
        force_frame = frame.iloc[0:0].copy()

    if eit_frame.empty:
        raise ValueError("No EIT rows found. Expected row_type baseline_eit/eit, or non-empty EIT columns.")

    force_value_column = next(
        (col for col in ["force_N", "force_sample_N", "latest_force_N", "actual_force_N"] if col in frame.columns),
        None,
    )
    if force_value_column and not force_frame.empty:
        force_frame = force_frame.dropna(subset=[time_column, force_value_column]).sort_values(time_column)
        eit_frame = pd.merge_asof(
            eit_frame.sort_values(time_column),
            force_frame[[time_column, force_value_column]].rename(columns={force_value_column: "nearest_force_N"}),
            on=time_column,
            direction="nearest",
        )
    elif force_value_column and "nearest_force_N" not in eit_frame.columns:
        eit_frame["nearest_force_N"] = eit_frame[force_value_column]

    return eit_frame.sort_values(time_column), force_frame, eit_cols, time_column


data, force_data, eit_cols, default_time_column = prepare_repeated_contact_frames(raw_data)
eit_labels = [col.removeprefix("eit_") for col in eit_cols]
eit_col_by_index = {int(label): col for label, col in zip(eit_labels, eit_cols)}

row_type_summary = raw_data["row_type"].value_counts().to_dict() if "row_type" in raw_data.columns else "EIT-only CSV"
print(f"Loaded {len(data)} EIT rows with {len(eit_cols)} EIT channels")
print(f"Time column: {default_time_column}")
print(f"Row types: {row_type_summary}")

In [ ]:
# Choose the channel indices you want to plot against time.
# These are the numbers from the eit_<index> column names.
channel_indices = [0, 1, 2, 3]

analysis_phase = "dwell"  # set to None to include descend/dwell/retract/rest together
time_column = "phase_elapsed_s" if "phase_elapsed_s" in data.columns else default_time_column
colour_column = "bunch_index"
include_baseline_bunch = False

In [ ]:
def plot_eit_channels_against_time(
    frame,
    channel_indices,
    time_column="phase_elapsed_s",
    colour_column="bunch_index",
    phase=None,
    include_baseline_bunch=False,
    save=True,
):
    missing = [col for col in (time_column, colour_column) if col not in frame.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}")

    selected_cols = []
    for index in channel_indices:
        try:
            selected_cols.append(eit_col_by_index[int(index)])
        except KeyError as exc:
            raise KeyError(f"No EIT channel column found for index {index}") from exc

    plot_frame = frame.copy()
    if phase is not None and "phase" in plot_frame.columns:
        plot_frame = plot_frame[plot_frame["phase"].astype(str).str.lower() == str(phase).lower()]
    if not include_baseline_bunch:
        plot_frame = plot_frame[plot_frame[colour_column] != -1]
    plot_frame = plot_frame.dropna(subset=[time_column, colour_column, *selected_cols])

    if plot_frame.empty:
        phase_message = f" for phase {phase!r}" if phase is not None else ""
        raise ValueError(f"No EIT rows available to plot{phase_message}.")

    bunches = sorted(plot_frame[colour_column].dropna().unique())
    cmap = plt.get_cmap("tab20", max(len(bunches), 1))
    colours = {bunch: cmap(i) for i, bunch in enumerate(bunches)}

    fig, axes = plt.subplots(
        len(selected_cols),
        1,
        figsize=(12, max(3, 2.5 * len(selected_cols))),
        sharex=True,
        squeeze=False,
    )

    for ax, col, index in zip(axes.flat, selected_cols, channel_indices):
        for bunch in bunches:
            bunch_frame = plot_frame[plot_frame[colour_column] == bunch].sort_values(time_column)
            label = f"bunch {bunch}"
            if "target_z_mm" in bunch_frame.columns and bunch_frame["target_z_mm"].notna().any():
                target_z = bunch_frame["target_z_mm"].dropna().iloc[0]
                label = f"{label}, z={target_z:g} mm"
            ax.plot(
                bunch_frame[time_column],
                bunch_frame[col],
                marker="o",
                markersize=2.5,
                linewidth=1,
                color=colours[bunch],
                label=label,
            )
        ax.set_ylabel(f"eit_{int(index)} (V)")
        ax.grid(True, alpha=0.25)

    axes.flat[-1].set_xlabel(time_column)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, title=colour_column, loc="center right")
        fig.subplots_adjust(right=0.78)

    phase_title = f", phase={phase}" if phase is not None else ""
    fig.suptitle(f"Selected EIT channels against time, coloured by contact{phase_title}", y=1.0)
    fig.tight_layout()

    if save:
        channel_tag = "_".join(str(int(index)) for index in channel_indices)
        phase_tag = f"_{phase}" if phase is not None else "_all-phases"
        output_path = Path("../../Figures") / f"EITTimeByBunch_{Path(csv_path).stem}{phase_tag}_ch{channel_tag}.svg"
        fig.savefig(output_path, format="svg", bbox_inches="tight")
        print(f"Saved {output_path}")

    return fig, axes

In [ ]:
plot_eit_channels_against_time(
    data,
    channel_indices,
    time_column=time_column,
    colour_column=colour_column,
    phase=analysis_phase,
    include_baseline_bunch=include_baseline_bunch,
)
plt.show()